In [ ]:
!pip install pandas

In [11]:
import json
import psycopg2
from pathlib import Path


DB_CONFIG = {
    "host": "localhost",
    "port": 55432,
    "database": "supply_chain",
    "user": "supplychain_app",
    "password": "Cr7@1034"
}


OUTPUT_FILE = Path(
    "output/database_schema.json"
)



def get_connection():

    return psycopg2.connect(
        **DB_CONFIG
    )



def get_tables(cursor):

    cursor.execute(
        """
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema='public'
        AND table_type='BASE TABLE'
        ORDER BY table_name;
        """
    )

    return [
        row[0]
        for row in cursor.fetchall()
    ]



def get_columns(cursor, table_name):

    cursor.execute(
        """
        SELECT

            column_name,

            data_type,

            character_maximum_length,

            is_nullable,

            column_default

        FROM information_schema.columns

        WHERE table_schema='public'

        AND table_name=%s

        ORDER BY ordinal_position;

        """,
        (
            table_name,
        )
    )


    return cursor.fetchall()



def get_primary_keys(cursor, table_name):

    cursor.execute(
        """
        SELECT
            kcu.column_name

        FROM
            information_schema.table_constraints tc

        JOIN
            information_schema.key_column_usage kcu

        ON
            tc.constraint_name=kcu.constraint_name

        WHERE
            tc.table_name=%s

        AND
            tc.constraint_type='PRIMARY KEY';

        """,
        (
            table_name,
        )
    )


    return [
        row[0]
        for row in cursor.fetchall()
    ]



def get_foreign_keys(cursor, table_name):

    cursor.execute(
        """
        SELECT

            kcu.column_name,

            ccu.table_name AS referenced_table,

            ccu.column_name AS referenced_column


        FROM
            information_schema.table_constraints tc


        JOIN
            information_schema.key_column_usage kcu

        ON
            tc.constraint_name=kcu.constraint_name


        JOIN
            information_schema.constraint_column_usage ccu

        ON
            ccu.constraint_name=tc.constraint_name


        WHERE
            tc.constraint_type='FOREIGN KEY'

        AND
            tc.table_name=%s;

        """,
        (
            table_name,
        )
    )


    return cursor.fetchall()



def build_create_table(cursor, table):


    columns=get_columns(
        cursor,
        table
    )


    primary_keys=get_primary_keys(
        cursor,
        table
    )


    foreign_keys=get_foreign_keys(
        cursor,
        table
    )



    ddl=[]


    ddl.append(
        f"CREATE TABLE {table} ("
    )



    column_lines=[]


    for column in columns:


        name=column[0]

        dtype=column[1]

        length=column[2]

        nullable=column[3]

        default=column[4]



        if length:

            dtype=f"{dtype}({length})"



        line=f"    {name} {dtype}"



        if nullable=="NO":

            line+=" NOT NULL"



        if default:

            line+=f" DEFAULT {default}"



        column_lines.append(
            line
        )



    if primary_keys:

        column_lines.append(

            "    PRIMARY KEY ("
            +
            ", ".join(primary_keys)
            +
            ")"

        )



    for fk in foreign_keys:


        column_lines.append(

            f"    FOREIGN KEY ({fk[0]}) "
            f"REFERENCES {fk[1]}({fk[2]})"

        )



    ddl.append(
        ",\n".join(column_lines)
    )


    ddl.append(
        ");"
    )


    return "\n".join(ddl)




def main():

    conn=get_connection()

    cursor=conn.cursor()


    try:


        tables=get_tables(
            cursor
        )


        schema_json={}



        for table in tables:


            print(
                "Exporting:",
                table
            )


            schema_json[table]=build_create_table(
                cursor,
                table
            )



        with open(
            OUTPUT_FILE,
            "w",
            encoding="utf-8"
        ) as f:


            json.dump(
                schema_json,
                f,
                indent=4
            )



        print(
            "\nSchema exported:",
            OUTPUT_FILE
        )



    finally:

        cursor.close()

        conn.close()



if __name__=="__main__":

    main()

Exporting: customers
Exporting: event_outbox
Exporting: inventory
Exporting: inventory_allocations
Exporting: inventory_reservations
Exporting: inventory_snapshots
Exporting: order_items
Exporting: orders
Exporting: payments
Exporting: products
Exporting: purchase_order_items
Exporting: purchase_orders
Exporting: shipment_items
Exporting: shipments
Exporting: suppliers
Exporting: warehouse_locations
Exporting: warehouse_tasks
Exporting: warehouses
Exporting: worker_productivity
Exporting: workers

Schema exported: output\database_schema.json
